In [ ]:
import tensorflow as tf
from tensorflow import keras
import numpy as np
from sklearn.model_selection import KFold
import pandas as pd
import pandas as pd
import polars as pl
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import *
from tensorflow.keras.models import Model
from sklearn.metrics import r2_score
import gc
import matplotlib.pyplot as plt


def reduce_mem_usage(df: pd.DataFrame, float16_as32: bool = True) -> pd.DataFrame:
        start_mem = df.memory_usage().sum() / 1024**2
        print(f"Memory usage of dataframe is {start_mem:.2f} MB")
        for col in df.columns:
            col_type = df[col].dtype
            if col_type != object and str(col_type) != "category":
                c_min, c_max = df[col].min(), df[col].max()
                if str(col_type)[:3] == "int":
                    if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                        df[col] = df[col].astype(np.int8)
                    elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                        df[col] = df[col].astype(np.int16)
                    elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                        df[col] = df[col].astype(np.int32)
                    elif c_min > np.iinfo(np.int64).min and c_max < np.iinfo(np.int64).max:
                        df[col] = df[col].astype(np.int64)
                else:
                    if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
                        df[col] = df[col].astype(np.float32) if float16_as32 else df[col].astype(np.float16)
                    elif c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                        df[col] = df[col].astype(np.float32)
                    else:
                        df[col] = df[col].astype(np.float64)
        end_mem = df.memory_usage().sum() / 1024**2
        print(f"Memory usage after optimization is: {end_mem:.2f} MB")
        print(f"Decreased by {100 * (start_mem - end_mem) / start_mem:.1f}%")
        return df

class CONFIG:
    target_col = "responder_6"
    lag_cols_original = ["date_id", "symbol_id"] + [f"responder_{idx}" for idx in range(9)]
    lag_cols_rename = { f"responder_{idx}" : f"responder_{idx}_lag_1" for idx in range(9)}
    lag_cols = [f"responder_{idx}_lag_1" for idx in range(9)]
    windows = [484, 968]  # Half day, full day
    
    adaptive_windows = [242, 484]
    
    created_features_names = (
        [f"{target_col}_{suffix}" for target_col in lag_cols 
         for suffix in [f"rolling_mean_{window}" for window in [484, 968]] +
                       [f"rolling_std_{window}" for window in [484, 968]] +
                       #[f"ewm_mean_{window}" for window in windows] +
                       [f"rolling_mean_diff_{window}" for window in [242, 484]] +
                       [f"rolling_std_ratio_{window}" for window in [242, 484]] +
                       [f"rolling_quantile_{window}" for window in [242, 484]]
        ]
    )
    time_features_names = ["sin_date_id", "cos_date_id", "sin_time_id", "cos_time_id"]
    valid_date = 1660
    features_names = [f'feature_{idx:02d}' for idx in range(79)] + [f'responder_{idx}_lag_1' for idx in range(9)] + time_features_names #+ ['symbol_id'] + created_features_names
    target_name = 'responder_6'
    weight_name = 'weight'

def create_features(lags):
    """Enhanced feature creation with focus on non-stationarity and fat tails"""
    for col in CONFIG.lag_cols:
        for window in CONFIG.windows:
            lags = lags.with_columns([
                pl.col(col).rolling_mean(window_size=window).over(['symbol_id']).alias(f'{col}_rolling_mean_{window}'),
                pl.col(col).rolling_std(window_size=window).over(['symbol_id']).alias(f'{col}_rolling_std_{window}'),
            ])
            
            # Exponential weighted mean with dynamic adjustment
            #weights = np.exp(np.linspace(-2., 0., window))  # Increased decay for faster adaptation
            #weights /= weights.sum()
            #lags = lags.with_columns([
            #    pl.col(col).ewm_mean(
            #        window_size=window,
            #        weights=weights
            #    ).over(['symbol_id']).alias(f'{col}_ewm_mean_{window}')
            #])
        
        for window in CONFIG.adaptive_windows:
            lags = lags.with_columns([
                (pl.col(col).rolling_mean(window_size=window).over(['symbol_id']) - 
                 pl.col(col).rolling_mean(window_size=window*2).over(['symbol_id']))
                .alias(f'{col}_rolling_mean_diff_{window}'),
                
                # Ratio of short-term to long-term volatility for regime changes
                (pl.col(col).rolling_std(window_size=window).over(['symbol_id']) /
                 pl.col(col).rolling_std(window_size=window*2).over(['symbol_id']))
                .alias(f'{col}_rolling_std_ratio_{window}'),
                
                pl.col(col).rolling_quantile(window_size=window, quantile=0.75)
                .over(['symbol_id']).alias(f'{col}_rolling_quantile_{window}')
            ])
    
    return lags


base_dir = "/kaggle/input/jane-street-real-time-market-data-forecasting/train.parquet/"
partitions = [f"{base_dir}/partition_id={i}/part-0.parquet" for i in range(8, 10)]
train = pl.concat([pl.read_parquet(part) for part in partitions])

# Create lags
lags = train.select(pl.col(CONFIG.lag_cols_original))
lags = lags.rename(CONFIG.lag_cols_rename)
lags = lags.with_columns(
    date_id = pl.col('date_id') + 1,  # lagged by 1 day
)

#lags = create_features(lags)

lags = lags.group_by(["date_id", "symbol_id"], maintain_order=True).last()
train = train.join(lags, on=["date_id", "symbol_id"], how="left")
del lags
gc.collect()

# Add time features
train = train.with_columns([
    (2 * np.pi * pl.col('date_id') / 252).sin().alias('sin_date_id'),
    (2 * np.pi * pl.col('date_id') / 252).cos().alias('cos_date_id'),
    (2 * np.pi * pl.col('time_id') / 967).sin().alias('sin_time_id'),
    (2 * np.pi * pl.col('time_id') / 967).cos().alias('cos_time_id')
])

# Fill NA values
train = train.fill_null(strategy="forward")
train = train.drop_nulls()

# Split into train and validation
val = train.filter(pl.col('date_id') >= CONFIG.valid_date)
train = train.filter(pl.col('date_id') < CONFIG.valid_date)

# Create final datasets in pandas
X_train = train.select(CONFIG.features_names).to_pandas()
y_train = train.select(CONFIG.target_name).to_pandas()
weight_train = train.select(CONFIG.weight_name).to_pandas()
del train
gc.collect()

X_val = val.select(CONFIG.features_names).to_pandas()
y_val = val.select(CONFIG.target_name).to_pandas()
weight_val = val.select(CONFIG.weight_name).to_pandas()
del val
gc.collect()

# Apply memory reduction if needed
X_train = reduce_mem_usage(X_train)
X_val = reduce_mem_usage(X_val)
y_train['responder_6']=y_train['responder_6'].astype('float16')
y_val['responder_6']=y_val['responder_6'].astype('float16')

In [ ]:
y_train['responder_6']=y_train['responder_6'].astype('float16')

In [ ]:
class custom_args():
    def __init__(self):
        self.usegpu = True
        self.gpuid = 0
        self.seed = 42
        self.model = 'nn'
        self.use_wandb = False
        self.project = 'js-xs-nn-with-lags'
        self.dname = "./input_df/"
        self.loader_workers = 4
        self.bs = 8192
        self.lr = 1e-3
        self.weight_decay = 5e-4
        self.dropouts = [0.1, 0.1]
        self.n_hidden = [512, 512, 256]
        self.patience = 25
        self.max_epochs = 2000
        self.N_fold = 5

class R2Score(keras.metrics.Metric):
    def __init__(self, name='r2_score', **kwargs):
        super().__init__(name=name, **kwargs)
        self.weighted_squared_sum = self.add_weight(name='wss', initializer='zeros')
        self.weighted_sum = self.add_weight(name='ws', initializer='zeros')
        self.count = self.add_weight(name='count', initializer='zeros')

    def update_state(self, y_true, y_pred, sample_weight=None):
        if sample_weight is None:
            sample_weight = tf.ones_like(y_true)
        
        weighted_squared_sum = tf.reduce_sum(sample_weight * tf.square(y_true))
        weighted_sum = tf.reduce_sum(sample_weight * tf.square(y_pred - y_true))
        
        self.weighted_squared_sum.assign_add(weighted_squared_sum)
        self.weighted_sum.assign_add(weighted_sum)
        self.count.assign_add(tf.reduce_sum(sample_weight))

    def result(self):
        return 1.0 - self.weighted_sum / (self.weighted_squared_sum + 1e-38)

    def reset_state(self):
        self.weighted_squared_sum.assign(0.0)
        self.weighted_sum.assign(0.0)
        self.count.assign(0.0)

def create_model(input_dim, hidden_dims=[512, 512, 256], dropouts=[0.1, 0.1]):
    model = keras.Sequential()
    
    model.add(keras.layers.BatchNormalization(input_shape=(input_dim,)))
    model.add(keras.layers.Dropout(dropouts[0]))
    model.add(keras.layers.Dense(hidden_dims[0]))
    
    for i, hidden_dim in enumerate(hidden_dims[1:]):
        model.add(keras.layers.BatchNormalization())
        model.add(tf.keras.layers.Activation(tf.keras.activations.silu))  # Equivalent to PyTorch's SiLU
        if i < len(dropouts) - 1:
            model.add(keras.layers.Dropout(dropouts[i + 1]))
        model.add(keras.layers.Dense(hidden_dim))
    
    model.add(keras.layers.Dense(1))
    model.add(keras.layers.Activation('tanh'))
    model.add(keras.layers.Lambda(lambda x: 5 * x))
    
    return model

def train_model(X_train, X_val, y_train, y_val, weight_train, weight_val, config, args):
    tf.keras.mixed_precision.set_global_policy('mixed_float16')
    tf.random.set_seed(args.seed)
    
    models = []
    train_dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train, weight_train))
    train_dataset = train_dataset.shuffle(buffer_size=10000)
    train_dataset = train_dataset.batch(args.bs)
    train_dataset = train_dataset.prefetch(tf.data.AUTOTUNE)

    val_dataset = tf.data.Dataset.from_tensor_slices((X_val, y_val, weight_val))
    val_dataset = val_dataset.batch(args.bs)
    val_dataset = val_dataset.prefetch(tf.data.AUTOTUNE)
    
    for fold in range(args.N_fold):
        print(f'Training fold {fold + 1}/{args.N_fold}')
        
        model = create_model(
            input_dim=X_train.shape[1],
            hidden_dims=args.n_hidden,
            dropouts=args.dropouts
        )
        
        optimizer = keras.optimizers.Adam(
            learning_rate=args.lr,
            weight_decay=args.weight_decay
        )
        
        model.compile(
            optimizer=optimizer,
            loss=keras.losses.MeanSquaredError(),
            metrics=[R2Score()]
        )
        
        callbacks = [
            keras.callbacks.EarlyStopping(
                monitor='val_loss',
                patience=args.patience,
                mode='min',
                restore_best_weights=True,
                verbose=1
            ),
            keras.callbacks.ReduceLROnPlateau(
                monitor='val_loss',
                factor=0.5,
                patience=5,
                mode='min',
                verbose=1
            ),
            keras.callbacks.ModelCheckpoint(
                filepath=f'./models/keras_model_{fold}.keras',
                monitor='val_loss',
                mode='min',
                save_best_only=True,
                verbose=1
            )
        ]
        
        history = model.fit(
            train_dataset,
            validation_data=val_dataset,
            epochs=args.max_epochs,
            callbacks=callbacks,
            verbose=1
        )
        
        models.append(model)
        
        val_loss = min(history.history['val_loss'])
        val_r2 = max(history.history['val_r2_score'])
        print(f'Fold {fold + 1} - Best val_loss: {val_loss:.6f}, Best val_r2: {val_r2:.6f}')
        
    return models

def predict(models, X_test):
    y_pred = np.zeros(len(X_test))
    for model in models:
        y_pred += model.predict(X_test, batch_size=8192, verbose=0) / len(models)
    return y_pred

args = custom_args()

models = train_model(
    X_train=X_train,
    X_val=X_val,
    y_train=y_train,
    y_val=y_val,
    weight_train=weight_train,
    weight_val=weight_val,
    config=CONFIG,
    args=args
)

y_pred = predict(models, X_val)

In [ ]:
def r2_val(y_true, y_pred, sample_weight):
    r2 = 1 - np.average((y_pred - y_true) ** 2, weights=sample_weight) / (np.average((y_true) ** 2, weights=sample_weight) + 1e-38)
    return r2

valid_score = r2_score( y_val, y_pred, sample_weight=weight_val )
valid_score